In [8]:
import pandas as pd

# -----------------------
# Load Excel file
# -----------------------
file_path = "/mnt/f/Solar-Review-Ppaer/Monthwise-Data-Sheets/Data_clean/20. Dec-2025-filtered.xlsx"
df = pd.read_excel(file_path)

# -----------------------
# Clean column names
# -----------------------
df.columns = df.columns.str.strip()

# -----------------------
# Convert Time
# -----------------------
df["Time_dt"] = pd.to_datetime(
    df["Time"].astype(str),
    format="%H.%M.%S",
    errors="coerce"
).dt.time

# -----------------------
# Create DateTime column
# -----------------------
df["DateTime"] = pd.to_datetime(
    df["Year"].astype(str) + "-" +
    df["Month"].astype(str).str.zfill(2) + "-" +
    df["Date"].astype(str).str.zfill(2) + " " +
    df["Time"].astype(str).str.replace(".", ":", regex=False),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Remove invalid datetime rows
df = df.dropna(subset=["DateTime"])

# -----------------------
# Filter 6 AM to 6 PM
# -----------------------
df = df[
    (df["Time_dt"] >= pd.to_datetime("06:00:00").time()) &
    (df["Time_dt"] <= pd.to_datetime("18:00:00").time())
]

# -----------------------
# Set datetime index
# -----------------------
df = df.set_index("DateTime")

# -----------------------
# Check original data range
# -----------------------
print("Original data:")
print("From :", df.index.min())
print("To   :", df.index.max())
print("Rows :", len(df))

# -----------------------
# 10-minute averaging
# -----------------------
df_10min = (
    df
    .resample("10min")
    .mean(numeric_only=True)
    .dropna(how='all')
    .reset_index()
)

# -----------------------
# Check resampled data range
# -----------------------
print("\n10-minute data:")
print("From :", df_10min["DateTime"].min())
print("To   :", df_10min["DateTime"].max())
print("Rows :", len(df_10min))

# -----------------------
# Save output
# -----------------------
output_path = "/mnt/f/Solar-Review-Ppaer/Monthwise-Data-Sheets/Data_clean/10-Minute-Average-Data/20. Dec-2025-filtered-10-min-Avg.xlsx"

df_10min.to_excel(output_path, index=False)

print("\nSaved successfully:")
print(output_path)


Original data:
From : 2025-12-01 06:00:00
To   : 2025-12-31 18:00:00
Rows : 22210

10-minute data:
From : 2025-12-01 06:00:00
To   : 2025-12-31 18:00:00
Rows : 2263

Saved successfully:
/mnt/f/Solar-Review-Ppaer/Monthwise-Data-Sheets/Data_clean/10-Minute-Average-Data/20. Dec-2025-filtered-10-min-Avg.xlsx
